# 🚀 Huấn Luyện ViSEC Pitch-Fusion Baseline (ICASSP 2024) Trên Google Colab / Kaggle

Notebook này hướng dẫn chạy mã nguồn từ bài báo **ViSEC: A robust Pitch-fusion model for Speech Emotion Recognition in tonal languages (ICASSP 2024)** [GitHub thanhvp2102/ViSEC](https://github.com/thanhpv2102/ViSEC.git) trên tập dữ liệu được phân chia đồng bộ với **MaxMViT-MLP-SER**:
- **Phân chia dữ liệu (Stratified Split, Seed 42):**
  - Train: 80% (4,224 mẫu)
  - Validation: 10% (528 mẫu)
  - Test: 10% (528 mẫu)
- **4 Cảm xúc mục tiêu:** 0: `happy`, 1: `neutral`, 2: `sad`, 3: `angry`
- **Tự động đo lường:** Đánh giá trên tập **Test độc lập**, xuất `classification_report.txt`, `test_results.json` và vẽ **Confusion Matrix**.

### Bước 1: Kiểm tra cấu hình GPU
> Đảm bảo bạn đã chọn GPU (Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU).

In [ ]:
!nvidia-smi

### Bước 2: (Tùy chọn) Kết nối Google Drive để lưu checkpoint vĩnh viễn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ViSEC_Pitch_Checkpoints

### Bước 3: Lấy mã nguồn dự án

In [ ]:
# Nếu bạn chạy từ repository đã clone:
# %cd MaxMViT-MLP-SER

# Hoặc clone mới nhất từ GitHub:
# !git clone https://github.com/Huu2412/MaxMViT-MLP-SER.git
# %cd MaxMViT-MLP-SER

### Bước 4: Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q torch torchaudio transformers datasets accelerate soundfile librosa scikit-learn seaborn matplotlib pyyaml

### Bước 5: Bắt đầu Huấn Luyện Mô Hình Pitch-Fusion (ICASSP 2024)
Cấu hình đã tối ưu chống tràn bộ nhớ CUDA VRAM:
- `batch_size 2` + `grad_accum 4` (effective batch size: 8)
- `gradient_checkpointing` (giảm 60% activation memory)
- `max_duration 8.0` (tránh các file outlier quá dài gây OOM)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!python run_visec_baseline.py \
    --mode train_pitch \
    --batch_size 2 \
    --grad_accum 4 \
    --max_duration 8.0 \
    --gradient_checkpointing \
    --epochs 30 \
    --lr 1.5e-5 \
    --num_proc 4 \
    --seed 42 \
    --output_dir checkpoints/visec_pitch_baseline

### (Tùy chọn) Bước 5B: Huấn luyện Baseline Wav2Vec 2.0 Đơn Thuần (Không có Pitch)
Dùng để so sánh hiệu quả cải thiện của cơ chế Pitch Fusion so với Wav2Vec 2.0 gốc.

In [ ]:
# !python run_visec_baseline.py \
#     --mode train_no_joint \
#     --batch_size 2 \
#     --grad_accum 4 \
#     --epochs 30 \
#     --output_dir checkpoints/visec_no_joint_baseline

### Bước 6: Hiển thị Kết quả Đánh giá và Confusion Matrix trên Tập Test

In [ ]:
from IPython.display import Image, display
import json

cm_path = 'checkpoints/visec_pitch_baseline/confusion_matrix.png'
results_path = 'checkpoints/visec_pitch_baseline/test_results.json'

try:
    with open(results_path, 'r', encoding='utf-8') as f:
        res = json.load(f)
    print(f"=== TEST RESULTS ===")
    print(f"Macro-F1:         {res['macro_f1']*100:.2f}%")
    print(f"Balanced Acc:     {res['balanced_accuracy']*100:.2f}%")
    print(f"Overall Acc:      {res['accuracy']*100:.2f}%")
    print("\nClassification Report:")
    print(res['classification_report'])
    display(Image(cm_path))
except Exception as e:
    print(f"Chưa tìm thấy kết quả test: {e}")

### Bước 7: Sao lưu Checkpoint về Google Drive (Nếu cần)

In [ ]:
!cp -r checkpoints/visec_pitch_baseline /content/drive/MyDrive/ViSEC_Pitch_Checkpoints/
print("Đã sao lưu thành công sang Google Drive!")